In [1]:
# Cell 1: Imports + paths
from pathlib import Path
import os
import json
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

BASE_DIR = Path("..").resolve()
DATASET_PATH = BASE_DIR / "dataset" / "jan to may police violation_anonymized.csv"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR = BASE_DIR / "data"

MODELS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print("Ready")
print("Dataset:", DATASET_PATH)
print("Models :", MODELS_DIR)
print("Data   :", DATA_DIR)

Ready
Dataset: C:\MargDristi\dataset\jan to may police violation_anonymized.csv
Models : C:\MargDristi\models
Data   : C:\MargDristi\data


In [2]:
# Cell 2: Load & inspect
df = pd.read_csv(DATASET_PATH)

print(f"Total records: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst row sample:")
print(df.iloc[0])
print("\nNull counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Total records: 298450
Columns: ['id', 'latitude', 'longitude', 'location', 'vehicle_number', 'vehicle_type', 'description', 'violation_type', 'offence_code', 'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp']

First row sample:
id                                                                     FKID000000
latitude                                                                12.925557
longitude                                                               77.618665
location                        18th Main Road, Block 2, Koramangala, Bengalur...
vehicle_number                                                        FKN00GL0000
vehicle_type                                                                  CAR
description     

In [3]:
# Cell 3: Parse violation types
def parse_primary_violation(val):
    if pd.isna(val):
        return "UNKNOWN"
    try:
        parsed = json.loads(str(val).replace("'", '"'))
        return parsed[0] if parsed else "UNKNOWN"
    except:
        return str(val).strip("[]'\"")

def parse_violation_count(val):
    if pd.isna(val):
        return 1
    try:
        parsed = json.loads(str(val).replace("'", '"'))
        return len(parsed)
    except:
        return 1

df["primary_violation"] = df["violation_type"].apply(parse_primary_violation)
df["violation_count"] = df["violation_type"].apply(parse_violation_count)

print(df[["violation_type", "primary_violation", "violation_count"]].head())
print("\nViolation types:")
print(df["primary_violation"].value_counts().head(15))

                                   violation_type primary_violation  \
0  ["WRONG PARKING","PARKING NEAR ROAD CROSSING"]     WRONG PARKING   
1                                  ["NO PARKING"]        NO PARKING   
2      ["WRONG PARKING","PARKING IN A MAIN ROAD"]     WRONG PARKING   
3                                  ["NO PARKING"]        NO PARKING   
4                                  ["NO PARKING"]        NO PARKING   

   violation_count  
0                2  
1                1  
2                2  
3                1  
4                1  

Violation types:
primary_violation
WRONG PARKING                                 147493
NO PARKING                                    128624
PARKING IN A MAIN ROAD                         17084
PARKING ON FOOTPATH                             2447
DEFECTIVE NUMBER PLATE                           807
PARKING NEAR ROAD CROSSING                       646
PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC          594
DOUBLE PARKING                          

In [4]:
# Cell 4: Timestamp features
df["created_dt"] = pd.to_datetime(df["created_datetime"], utc=True, errors="coerce")
df["created_ist"] = df["created_dt"].dt.tz_convert("Asia/Kolkata")

df["hour"] = df["created_ist"].dt.hour
df["day_num"] = df["created_ist"].dt.dayofweek
df["day_name"] = df["created_ist"].dt.day_name()
df["month_name"] = df["created_ist"].dt.month_name()
df["date"] = df["created_ist"].dt.date

print(f"Date range (IST): {df['created_ist'].min()} → {df['created_ist'].max()}")
print("\nViolations by hour:")
print(df["hour"].value_counts().sort_index())

Date range (IST): 2023-11-10 00:41:46+05:30 → 2024-04-08 23:00:46+05:30

Violations by hour:
hour
0.0      5815
1.0     11098
2.0     16261
3.0     21565
4.0     23512
5.0     22193
6.0     19836
7.0     19445
8.0     25790
9.0     26994
10.0    32580
11.0    32176
12.0    19689
13.0    11546
14.0     5634
15.0     1224
16.0      583
17.0      377
18.0      150
19.0       27
20.0       42
21.0      148
22.0      725
23.0     1035
Name: count, dtype: int64


In [5]:
# Cell 5: CIS score
VIOLATION_SEVERITY = {
    "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS": 5.0,
    "PARKING NEAR ROAD CROSSING": 4.5,
    "DOUBLE PARKING": 4.5,
    "PARKING IN A MAIN ROAD": 4.0,
    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC": 3.5,
    "NO PARKING": 3.0,
    "WRONG PARKING": 2.5,
    "PARKING ON FOOTPATH": 2.0,
    "PARKING OPPOSITE TO ANOTHER PARKED VEHICLE": 2.5,
    "PARKING OTHER THAN BUS STOP": 2.0,
    "DEFECTIVE NUMBER PLATE": 1.5,
    "USING BLACK FILM/OTHER MATERIALS": 1.5,
    "REFUSE TO GO FOR HIRE": 1.5,
    "AGAINST ONE WAY/NO ENTRY": 1.5,
    "WITHOUT SIDE MIRROR": 1.0,
    "VIOLATING LANE DISIPLINE": 1.5,
    "FAIL TO USE SAFETY BELTS": 1.0,
    "OBSTRUCTING DRIVER": 1.0,
    "DEMANDING EXCESS FARE": 1.0,
    "H T V PROHIBITED": 1.0,
}

VEHICLE_SCORE = {
    "HGV": 3.0, "LORRY/GOODS VEHICLE": 3.0,
    "BUS (BMTC/KSRTC)": 2.8, "PRIVATE BUS": 2.8,
    "TOURIST BUS": 2.8, "FACTORY BUS": 2.8,
    "SCHOOL VEHICLE": 2.5, "TANKER": 2.5,
    "MAXI-CAB": 2.2, "TEMPO": 2.2,
    "MINI LORRY": 2.0, "LGV": 2.0,
    "TRACTOR": 2.0, "VAN": 1.8,
    "JEEP": 1.5, "CAR": 1.5,
    "GOODS AUTO": 1.3, "PASSENGER AUTO": 1.2,
    "MOTOR CYCLE": 1.0, "SCOOTER": 1.0,
    "MOPED": 1.0, "OTHERS": 1.2,
}

def compute_cis(row):
    vts = VIOLATION_SEVERITY.get(row["primary_violation"], 2.0)
    vcs = VEHICLE_SCORE.get(str(row["vehicle_type"]), 1.2)
    jpm = 1.4 if str(row.get("junction_name", "No Junction")) != "No Junction" else 1.0
    mlm = 1.4 if row["violation_count"] >= 3 else (1.2 if row["violation_count"] == 2 else 1.0)
    return round(vts * vcs * jpm * mlm, 4)

df["is_named_junction"] = (df["junction_name"] != "No Junction").astype(int)
df["cis_score"] = df.apply(compute_cis, axis=1)

print(df["cis_score"].describe())
print("\nTop CIS violation types:")
print(df.groupby("primary_violation")["cis_score"].mean().sort_values(ascending=False).head(10))

count    298450.000000
mean          4.523806
std           1.815619
min           1.200000
25%           3.500000
50%           4.200000
75%           5.250000
max          26.460000
Name: cis_score, dtype: float64

Top CIS violation types:
primary_violation
PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS     10.816800
DOUBLE PARKING                                 9.152032
PARKING NEAR ROAD CROSSING                     8.383848
PARKING IN A MAIN ROAD                         7.971402
PARKING OTHER THAN BUS STOP                    6.738969
PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC        6.016653
PARKING OPPOSITE TO ANOTHER PARKED VEHICLE     5.477463
NO PARKING                                     4.647263
FAIL TO USE SAFETY BELTS                       4.200000
H T V PROHIBITED                               4.009655
Name: cis_score, dtype: float64


In [6]:
# Cell 6: Clean dataset
keep_cols = [
    "latitude", "longitude",
    "junction_name", "is_named_junction",
    "created_ist", "hour", "day_num", "day_name", "month_name", "date",
    "vehicle_type", "primary_violation", "violation_count",
    "police_station", "device_id", "created_by_id",
    "cis_score"
]

df_clean = df[keep_cols].copy()
df_clean = df_clean.rename(columns={"created_ist": "timestamp"})

df_clean = df_clean[
    (df_clean["latitude"] != 0) &
    (df_clean["longitude"] != 0) &
    df_clean["latitude"].notna() &
    df_clean["longitude"].notna()
].reset_index(drop=True)

print(f"Clean dataset: {len(df_clean)} records")
print(df_clean.columns.tolist())

df_clean.to_csv(DATA_DIR / "violations_clean.csv", index=False)
print("Saved:", DATA_DIR / "violations_clean.csv")

Clean dataset: 298450 records
['latitude', 'longitude', 'junction_name', 'is_named_junction', 'timestamp', 'hour', 'day_num', 'day_name', 'month_name', 'date', 'vehicle_type', 'primary_violation', 'violation_count', 'police_station', 'device_id', 'created_by_id', 'cis_score']
Saved: C:\MargDristi\data\violations_clean.csv


In [7]:
# Cell 7: Quick check
peak_traffic = df_clean[df_clean["hour"].between(12, 17)]
rest = df_clean[~df_clean["hour"].between(12, 17)]

print("=== THE ENFORCEMENT BLINDSPOT ===")
print(f"Violations 12:00-17:59: {len(peak_traffic)} ({100*len(peak_traffic)/len(df_clean):.1f}%)")
print(f"Violations other hours: {len(rest)} ({100*len(rest)/len(df_clean):.1f}%)")

=== THE ENFORCEMENT BLINDSPOT ===
Violations 12:00-17:59: 39053 (13.1%)
Violations other hours: 259397 (86.9%)


In [8]:
# Cell 8: HDBSCAN clustering
import hdbscan
import joblib
import numpy as np

coords = df_clean[['latitude', 'longitude']].astype('float32').values

print(f"Running HDBSCAN on {len(coords)} GPS coordinates...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=100,
    min_samples=25,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

labels = clusterer.fit_predict(coords)
df_clean['cluster_id'] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())

print(f"\nFinal model: {n_clusters} clusters, {n_noise} noise points ({100*n_noise/len(labels):.1f}%)")

joblib.dump(clusterer, MODELS_DIR / 'hdbscan_model.pkl')
print("✅ Saved: hdbscan_model.pkl")

Running HDBSCAN on 298450 GPS coordinates...

Final model: 587 clusters, 65895 noise points (22.1%)
✅ Saved: hdbscan_model.pkl


In [9]:
# Cell 9: Cluster metadata & tier classification
records = []

valid_clusters = sorted([c for c in df_clean['cluster_id'].unique() if c != -1])

for cid in valid_clusters:
    sub = df_clean[df_clean['cluster_id'] == cid]

    total_cis = sub['cis_score'].sum()
    avg_cis = sub['cis_score'].mean()
    count = len(sub)
    peak_hour = int(sub['hour'].mode()[0])
    top_vehicle = sub['vehicle_type'].mode()[0]
    top_violation = sub['primary_violation'].mode()[0]
    centroid_lat = sub['latitude'].mean()
    centroid_lng = sub['longitude'].mean()

    junc_series = sub[sub['junction_name'] != 'No Junction']['junction_name']
    if len(junc_series) > 0:
        zone_name = junc_series.mode()[0]
    else:
        zone_name = f"Cluster {cid} ({centroid_lat:.4f}, {centroid_lng:.4f})"

    records.append({
        'cluster_id': int(cid),
        'zone_name': zone_name,
        'centroid_lat': round(float(centroid_lat), 6),
        'centroid_lng': round(float(centroid_lng), 6),
        'violation_count': int(count),
        'total_cis': round(float(total_cis), 2),
        'avg_cis': round(float(avg_cis), 4),
        'peak_hour': int(peak_hour),
        'top_vehicle': top_vehicle,
        'top_violation': top_violation,
    })

clusters_df = pd.DataFrame(records).sort_values('total_cis', ascending=False).reset_index(drop=True)

n1 = max(1, int(len(clusters_df) * 0.15))
n2 = max(1, int(len(clusters_df) * 0.35))

clusters_df['tier'] = 'Tier 3'
clusters_df.iloc[:n1, clusters_df.columns.get_loc('tier')] = 'Tier 1'
clusters_df.iloc[n1:n1+n2, clusters_df.columns.get_loc('tier')] = 'Tier 2'

clusters_df['risk_score'] = (
    (clusters_df['total_cis'] / clusters_df['total_cis'].max()) * 70 +
    (clusters_df['violation_count'] / clusters_df['violation_count'].max()) * 30
).round(1)

print("Top 15 clusters:")
print(clusters_df[['cluster_id', 'zone_name', 'violation_count', 'total_cis', 'tier', 'risk_score', 'peak_hour']].head(15))

clusters_df.to_csv(DATA_DIR / 'cluster_metadata.csv', index=False)
df_clean.to_csv(DATA_DIR / 'violations_clustered.csv', index=False)

print("✅ Saved: cluster_metadata.csv")
print("✅ Saved: violations_clustered.csv")

Top 15 clusters:
    cluster_id                                       zone_name  \
0          186                  Cluster 186 (12.9336, 77.6910)   
1          117                  Cluster 117 (13.0083, 77.6953)   
2           87                   Cluster 87 (12.9963, 77.6686)   
3          469                     BTP057 - Anand Rao Junction   
4          436                BTP211 - Central Street Junction   
5          545                      BTP058 - Subbanna Junction   
6          528                  BTP051 - Safina Plaza Junction   
7          491               BTP045 - Danvanthri Road Junction   
8          410      BTP070 - Cholurpalya Junction, Magadi Road   
9          130                  Cluster 130 (12.9397, 77.6956)   
10         558                         BTP040 - Elite Junction   
11         501                  BTP051 - Safina Plaza Junction   
12         568                         BTP040 - Elite Junction   
13         406  BTP011 - RR Kalyana Mantapa, Dr. Rajkumar R

In [10]:
# Cell 10: Temporal pattern export
hourly_city = df_clean.groupby('hour').size().reset_index(name='count')

pivot_matrix = pd.crosstab(df_clean['day_num'], df_clean['hour'])
pivot_matrix.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

top_cluster_ids = clusters_df.head(10)['cluster_id'].tolist()

cluster_daily = {}
for cid in top_cluster_ids:
    sub = df_clean[df_clean['cluster_id'] == cid]
    daily = sub.groupby('date').size().reset_index(name='violations')
    daily['date'] = pd.to_datetime(daily['date'])
    cluster_daily[cid] = daily

hourly_city.to_csv(DATA_DIR / 'temporal_hourly_city.csv', index=False)
pivot_matrix.to_csv(DATA_DIR / 'temporal_heatmap_matrix.csv')

print("Hourly distribution (city-wide):")
print(hourly_city.to_string(index=False))
print("✅ Saved: temporal_hourly_city.csv")
print("✅ Saved: temporal_heatmap_matrix.csv")

Hourly distribution (city-wide):
 hour  count
  0.0   5815
  1.0  11098
  2.0  16261
  3.0  21565
  4.0  23512
  5.0  22193
  6.0  19836
  7.0  19445
  8.0  25790
  9.0  26994
 10.0  32580
 11.0  32176
 12.0  19689
 13.0  11546
 14.0   5634
 15.0   1224
 16.0    583
 17.0    377
 18.0    150
 19.0     27
 20.0     42
 21.0    148
 22.0    725
 23.0   1035
✅ Saved: temporal_hourly_city.csv
✅ Saved: temporal_heatmap_matrix.csv


In [16]:
# Cell 11: Fast Prophet Forecasting (Top 5 Hotspots)

import pickle
from prophet import Prophet
import pandas as pd
import numpy as np

# Only top 5 hotspots
top_cluster_ids = clusters_df.head(5)['cluster_id'].tolist()

prophet_models = {}
forecast_rows = []

print(f"Training forecasts for {len(top_cluster_ids)} hotspots...\n")

for i, cid in enumerate(top_cluster_ids, start=1):

    daily = cluster_daily[cid]

    if len(daily) < 20:
        print(f"[{i}/5] Cluster {cid}: skipped (only {len(daily)} days)")
        continue

    prophet_df = pd.DataFrame({
        'ds': pd.to_datetime(daily['date']),
        'y': daily['violations']
    })

    try:

        model = Prophet(
            weekly_seasonality=True,
            yearly_seasonality=False,
            daily_seasonality=False,
            changepoint_prior_scale=0.05,
            interval_width=0.90
        )

        model.fit(prophet_df)

        future = model.make_future_dataframe(periods=7)
        forecast = model.predict(future)

        future7 = forecast.tail(7)[
            ['ds', 'yhat', 'yhat_lower', 'yhat_upper']
        ].copy()

        future7['cluster_id'] = cid

        forecast_rows.append(future7)

        with open(MODELS_DIR / f'prophet_cluster_{cid}.pkl', 'wb') as f:
            pickle.dump(model, f)

        prophet_models[cid] = model

        print(
            f"[{i}/5] Cluster {cid} "
            f"✓ Forecast Avg={future7['yhat'].mean():.1f}"
        )

    except Exception as e:
        print(f"[{i}/5] Cluster {cid} FAILED: {e}")

if len(forecast_rows) > 0:

    forecast_df = pd.concat(
        forecast_rows,
        ignore_index=True
    )

    forecast_df.to_csv(
        DATA_DIR / "prophet_forecasts.csv",
        index=False
    )

    print("\n✅ Saved: prophet_forecasts.csv")

print(f"✅ Saved {len(prophet_models)} Prophet models")

18:52:50 - cmdstanpy - INFO - Chain [1] start processing
18:52:51 - cmdstanpy - INFO - Chain [1] done processing
18:52:51 - cmdstanpy - INFO - Chain [1] start processing
18:52:51 - cmdstanpy - INFO - Chain [1] done processing
18:52:51 - cmdstanpy - INFO - Chain [1] start processing
18:52:51 - cmdstanpy - INFO - Chain [1] done processing
18:52:51 - cmdstanpy - INFO - Chain [1] start processing
18:52:51 - cmdstanpy - INFO - Chain [1] done processing
18:52:51 - cmdstanpy - INFO - Chain [1] start processing
18:52:51 - cmdstanpy - INFO - Chain [1] done processing


In [12]:
print("Notebook is responsive")

In [17]:
# Cell 12: Enforcement simulator
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

np.random.seed(42)
rows = []

for _, cluster in clusters_df.iterrows():
    cid = int(cluster['cluster_id'])
    tier = cluster['tier']
    daily_avg = float(cluster['violation_count']) / 150.0

    for num_officers in range(1, 6):
        for hour in [6, 8, 10, 12, 14, 16, 18, 20, 22]:
            base_rate = 0.20 + (num_officers * 0.11)
            base_rate = min(base_rate, 0.72)

            hour_mult = 1.10 if 8 <= hour <= 18 else 0.85
            tier_mult = {'Tier 1': 0.85, 'Tier 2': 1.00, 'Tier 3': 1.15}.get(tier, 1.0)

            rate = base_rate * hour_mult * tier_mult
            rate = float(np.clip(rate + np.random.normal(0, 0.04), 0.05, 0.80))

            rows.append({
                'cluster_id': cid,
                'num_officers': num_officers,
                'hour': hour,
                'daily_avg': daily_avg,
                'tier1': 1 if tier == 'Tier 1' else 0,
                'tier2': 1 if tier == 'Tier 2' else 0,
                'peak_hour': 1 if 8 <= hour <= 18 else 0,
                'prevention_rate': rate,
            })

sim_df = pd.DataFrame(rows)

X = sim_df[['cluster_id', 'num_officers', 'hour', 'daily_avg', 'tier1', 'tier2', 'peak_hour']].values
y = sim_df['prevention_rate'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sim_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
sim_model.fit(X_train, y_train)

y_pred = sim_model.predict(X_test)
print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

joblib.dump(sim_model, MODELS_DIR / 'simulator_model.pkl')
sim_df.to_csv(DATA_DIR / 'simulator_training_data.csv', index=False)

print("✅ Saved: simulator_model.pkl")
print("✅ Saved: simulator_training_data.csv")

In [14]:
# Cell 13: Final verification
required_files = [
    DATA_DIR / 'violations_clean.csv',
    DATA_DIR / 'violations_clustered.csv',
    DATA_DIR / 'cluster_metadata.csv',
    DATA_DIR / 'temporal_hourly_city.csv',
    DATA_DIR / 'temporal_heatmap_matrix.csv',
    DATA_DIR / 'prophet_forecasts.csv',
    MODELS_DIR / 'hdbscan_model.pkl',
    MODELS_DIR / 'simulator_model.pkl',
]

print("=== FILE CHECKLIST ===")
for f in required_files:
    print(("✅" if f.exists() else "❌"), f.name, f.stat().st_size // 1024 if f.exists() else 0, "KB")

print("\nProphet models:")
for f in sorted(MODELS_DIR.glob("prophet_cluster_*.pkl")):
    print("✅", f.name, f.stat().st_size // 1024, "KB")

In [19]:
import os

print("Forecast CSV exists:",
      os.path.exists(DATA_DIR / "prophet_forecasts.csv"))

print("\nModels:")

for f in MODELS_DIR.glob("prophet_cluster_*.pkl"):
    print(f.name)

In [21]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

df_clean = pd.read_csv(
    DATA_DIR / "violations_clean.csv"
)

print(df_clean.shape)
print(df_clean.columns.tolist())

In [24]:
df_clean = df_clean.dropna(
    subset=[
        'hour',
        'latitude',
        'longitude',
        'cis_score',
        'is_named_junction'
    ]
).reset_index(drop=True)

print(len(df_clean))

In [25]:
from sklearn.preprocessing import StandardScaler
import hdbscan
import joblib

X = df_clean[
    [
        'latitude',
        'longitude',
        'hour',
        'cis_score',
        'is_named_junction'
    ]
].copy()

X_scaled = StandardScaler().fit_transform(X)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=50,
    cluster_selection_method='eom'
)

labels = clusterer.fit_predict(X_scaled)

df_clean['cluster_id'] = labels

clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = int((labels == -1).sum())

print(f"Clusters: {clusters}")
print(f"Noise points: {noise}")
print(f"Noise %: {100*noise/len(labels):.2f}")

joblib.dump(
    clusterer,
    MODELS_DIR / "hdbscan_model.pkl"
)

df_clean.to_csv(
    DATA_DIR / "violations_clustered.csv",
    index=False
)

print("✅ Saved improved HDBSCAN model")

In [26]:
from sklearn.preprocessing import StandardScaler
import hdbscan
import pandas as pd

X = df_clean[
    [
        'latitude',
        'longitude',
        'hour',
        'cis_score',
        'is_named_junction'
    ]
].copy()

X_scaled = StandardScaler().fit_transform(X)

results = []

for min_cluster_size in [100, 150, 200, 300]:
    for min_samples in [25, 50, 75]:

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_method='eom'
        )

        labels = clusterer.fit_predict(X_scaled)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        noise_pct = (
            (labels == -1).sum()
            / len(labels)
        ) * 100

        results.append({
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "clusters": n_clusters,
            "noise_pct": round(noise_pct, 2)
        })

results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        ['noise_pct', 'clusters']
    )
)

KeyboardInterrupt: 

In [27]:
from pathlib import Path
import pandas as pd
import joblib

BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

print("="*60)
print("FILE CHECK")
print("="*60)

required_files = [
    DATA_DIR / "violations_clean.csv",
    DATA_DIR / "violations_clustered.csv",
    DATA_DIR / "cluster_metadata.csv",
    DATA_DIR / "temporal_hourly_city.csv",
    DATA_DIR / "temporal_heatmap_matrix.csv",
    DATA_DIR / "prophet_forecasts.csv",
    DATA_DIR / "simulator_training_data.csv",
    MODELS_DIR / "hdbscan_model.pkl",
    MODELS_DIR / "simulator_model.pkl",
]

for f in required_files:
    if f.exists():
        print(f"✅ {f.name:<35} {round(f.stat().st_size/1024,2)} KB")
    else:
        print(f"❌ {f.name}")

print("\n")

print("="*60)
print("CLUSTER ANALYSIS")
print("="*60)

clusters_df = pd.read_csv(
    DATA_DIR / "cluster_metadata.csv"
)

violations_df = pd.read_csv(
    DATA_DIR / "violations_clustered.csv"
)

print("Total Hotspots:", len(clusters_df))

if 'cluster_id' in violations_df.columns:

    valid_clusters = (
        violations_df['cluster_id'] != -1
    ).sum()

    noise_points = (
        violations_df['cluster_id'] == -1
    ).sum()

    total_points = len(violations_df)

    print("Total Violations:", total_points)
    print("Clustered Points:", valid_clusters)
    print("Noise Points:", noise_points)
    print(
        "Noise Percentage:",
        round(noise_points/total_points*100,2),
        "%"
    )

print("\nTop 10 Hotspots")

cols = [
    c for c in [
        'cluster_id',
        'risk_score',
        'violation_count',
        'total_cis',
        'peak_hour'
    ]
    if c in clusters_df.columns
]

print(
    clusters_df[cols]
    .head(10)
    .to_string(index=False)
)

print("\n")

print("="*60)
print("FORECAST CHECK")
print("="*60)

forecast_df = pd.read_csv(
    DATA_DIR / "prophet_forecasts.csv"
)

print(
    "Forecast Rows:",
    len(forecast_df)
)

print(
    "Forecast Clusters:",
    forecast_df['cluster_id']
    .nunique()
)

print(
    forecast_df.head()
)

print("\n")

print("="*60)
print("PROPHET MODELS")
print("="*60)

prophet_models = list(
    MODELS_DIR.glob(
        "prophet_cluster_*.pkl"
    )
)

print(
    "Total Prophet Models:",
    len(prophet_models)
)

for f in sorted(prophet_models):
    print(
        f.name,
        round(f.stat().st_size/1024,2),
        "KB"
    )

print("\n")

print("="*60)
print("SIMULATOR MODEL")
print("="*60)

sim_model = joblib.load(
    MODELS_DIR /
    "simulator_model.pkl"
)

print(type(sim_model))

print("\n")

print("="*60)
print("FINAL SUMMARY")
print("="*60)

print(
    f"Hotspots Found: {len(clusters_df)}"
)

print(
    f"Forecast Models: {len(prophet_models)}"
)

print(
    f"Forecast Rows: {len(forecast_df)}"
)

print(
    f"Clustered Dataset Rows: {len(violations_df)}"
)

print("\nDONE")